# Piper TTS Training — Iu Mienh Voice

**Setup:** Upload `iu-mienh-tts.tar.gz` as a Kaggle Dataset, enable GPU, turn on Internet.

In [ ]:
# Cell 1: Find our dataset
import os, glob
print('Looking for dataset...')
!find /kaggle/input -maxdepth 4 -name 'metadata.txt' -o -name 'wavs' -type d 2>/dev/null
print('---')
!ls /kaggle/input/ 2>/dev/null
!ls /kaggle/input/*/ 2>/dev/null | head -10

In [ ]:
# Cell 2: Install dependencies
import sys, os
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'
print(f'Python {sys.version}')

!pip install -q pytorch-lightning==1.9.5
!pip install -q onnxruntime librosa tensorboard
!pip install -q cython

# Clone piper for training code (VitsModel) and build monotonic align
!rm -rf /tmp/piper
!git clone --depth 1 https://github.com/rhasspy/piper.git /tmp/piper
!cd /tmp/piper/src/python && bash build_monotonic_align.sh

# Add to path
sys.path.insert(0, '/tmp/piper/src/python')
os.environ['PYTHONPATH'] = '/tmp/piper/src/python'

from piper_train.vits.lightning import VitsModel
print('\n✅ Dependencies installed')

In [ ]:
# Cell 3: Prepare dataset
import sys
sys.path.insert(0, '/tmp/piper/src/python')
from pathlib import Path

# Auto-find metadata.txt wherever Kaggle put it
hits = list(Path('/kaggle/input').glob('**/metadata.txt'))
if not hits:
    raise FileNotFoundError(
        'metadata.txt not found! Check Cell 1 output for actual paths.'
    )

metadata_src = hits[0]
wavs_src = metadata_src.parent / 'wavs'
print(f'Metadata: {metadata_src}')
print(f'Wavs dir: {wavs_src}')
assert wavs_src.is_dir(), f'wavs/ not found at {wavs_src}'

# Setup working dirs
WORK = Path('/kaggle/working/iu-mienh')
DATASET = WORK / 'dataset'
OUTPUT = WORK / 'output'
DATASET.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

# Symlink wavs
wavs_link = DATASET / 'wavs'
if wavs_link.exists() or wavs_link.is_symlink():
    wavs_link.unlink()
wavs_link.symlink_to(wavs_src)

# Write filtered metadata.csv
lines = metadata_src.read_text().strip().split('\n')
valid = [l for l in lines if (wavs_src / l.split('|')[0]).exists()]
meta_csv = DATASET / 'metadata.csv'
meta_csv.write_text('\n'.join(valid) + '\n')

print(f'\n✅ {len(valid)} utterances ready ({len(lines)-len(valid)} skipped)')
for l in valid[:3]:
    p = l.split('|')
    print(f'  {p[0]} | {p[1][:50]}')

In [ ]:
# Cell 4: Preprocess — custom (bypasses piper_phonemize which won't build on Py3.12)
# Generates: config.json, dataset.jsonl, and cached .pt audio/spectrogram files
import json, csv, sys
from pathlib import Path
from hashlib import sha256
from collections import Counter
import torch
import librosa
import numpy as np

sys.path.insert(0, '/tmp/piper/src/python')
from piper_train.vits.mel_processing import spectrogram_torch

WORK = Path('/kaggle/working/iu-mienh')
DATASET = WORK / 'dataset'
PREPROC = WORK / 'preprocessed'
CACHE = PREPROC / 'cache' / '22050'
PREPROC.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 22050
FILTER_LENGTH = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024

# Read metadata.csv
meta_path = DATASET / 'metadata.csv'
wavs_dir = DATASET / 'wavs'

utterances = []
all_chars = Counter()
skipped = 0

with open(meta_path, 'r') as f:
    reader = csv.reader(f, delimiter='|')
    rows = list(reader)

print(f'Processing {len(rows)} utterances...')

for idx, row in enumerate(rows):
    if len(row) < 2:
        continue
    filename, text = row[0], row[-1]
    wav_path = wavs_dir / filename
    if not wav_path.exists():
        wav_path = wavs_dir / f'{filename}.wav'
    if not wav_path.exists():
        skipped += 1
        continue

    # Cache ID based on file path
    cache_id = sha256(str(wav_path.absolute()).encode()).hexdigest()
    norm_path = CACHE / f'{cache_id}.pt'
    spec_path = CACHE / f'{cache_id}.spec.pt'

    # Generate .pt files if not cached
    if not norm_path.exists() or not spec_path.exists():
        try:
            audio, _ = librosa.load(str(wav_path), sr=SAMPLE_RATE)
            audio_tensor = torch.FloatTensor(audio).unsqueeze(0)
            torch.save(audio_tensor, norm_path)

            spec = spectrogram_torch(
                audio_tensor, FILTER_LENGTH, SAMPLE_RATE,
                HOP_LENGTH, WIN_LENGTH, center=False
            ).squeeze(0)
            torch.save(spec, spec_path)
        except Exception as e:
            skipped += 1
            continue

    # Character-level phonemes
    phonemes = list(text)
    all_chars.update(phonemes)
    utterances.append({
        'text': text,
        'audio_path': str(wav_path),
        'audio_norm_path': str(norm_path),
        'audio_spec_path': str(spec_path),
        'phonemes': phonemes,
    })

    if (idx + 1) % 500 == 0:
        print(f'  {idx+1}/{len(rows)} processed...')

print(f'\nProcessed {len(utterances)} utterances (skipped {skipped})')
print(f'Unique characters: {len(all_chars)}')

# Build phoneme_id_map (char -> id)
# Reserve: 0=pad(_), 1=bos(^), 2=eos($)
sorted_chars = ['_', '^', '$'] + sorted(all_chars.keys())
phoneme_id_map = {c: i for i, c in enumerate(sorted_chars)}

# Assign phoneme_ids
for utt in utterances:
    ids = [phoneme_id_map['^']]
    for p in utt['phonemes']:
        if p in phoneme_id_map:
            ids.append(phoneme_id_map[p])
    ids.append(phoneme_id_map['$'])
    utt['phoneme_ids'] = ids

# Write config.json
config = {
    'dataset': 'iu-mienh',
    'audio': {'sample_rate': SAMPLE_RATE, 'quality': 'medium'},
    'espeak': {'voice': 'ium'},
    'language': {'code': 'ium'},
    'inference': {'noise_scale': 0.667, 'length_scale': 1, 'noise_w': 0.8},
    'phoneme_type': 'text',
    'phoneme_map': {},
    'phoneme_id_map': phoneme_id_map,
    'num_symbols': len(sorted_chars),
    'num_speakers': 1,
    'speaker_id_map': {},
    'piper_version': '1.0.0',
}

with open(PREPROC / 'config.json', 'w') as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

# Write dataset.jsonl
with open(PREPROC / 'dataset.jsonl', 'w') as f:
    for utt in utterances:
        # Only write fields the training expects
        out = {
            'phoneme_ids': utt['phoneme_ids'],
            'audio_norm_path': utt['audio_norm_path'],
            'audio_spec_path': utt['audio_spec_path'],
            'text': utt['text'],
        }
        json.dump(out, f, ensure_ascii=False)
        f.write('\n')

print(f'\n✅ Preprocessing complete!')
print(f'   {len(utterances)} utterances')
print(f'   {len(sorted_chars)} symbols')
print(f'   Config: {PREPROC / "config.json"}')
print(f'   Dataset: {PREPROC / "dataset.jsonl"}')

In [ ]:
# Cell 5: Train
import sys, os
sys.path.insert(0, '/tmp/piper/src/python')
os.environ['PYTHONPATH'] = '/tmp/piper/src/python'
from pathlib import Path

WORK = Path('/kaggle/working/iu-mienh')
PREPROC = WORK / 'preprocessed'
OUTPUT = WORK / 'output'

# Check for existing checkpoint to resume
ckpts = sorted(OUTPUT.glob('**/*.ckpt'))
resume = ''
if ckpts:
    resume = f'--resume_from_checkpoint {ckpts[-1]}'
    print(f'Resuming from: {ckpts[-1]}')
else:
    print('Starting fresh training')

MAX_EPOCHS = 100
BATCH = 16

!PYTHONPATH=/tmp/piper/src/python python3 -m piper_train \
    --dataset-dir {PREPROC} \
    --accelerator gpu \
    --devices 1 \
    --batch-size {BATCH} \
    --validation-split 0.05 \
    --max_epochs {MAX_EPOCHS} \
    --checkpoint-epochs 25 \
    --quality medium \
    --precision 32 \
    --default_root_dir {OUTPUT} \
    {resume}

In [ ]:
# Cell 6: Export to ONNX
import sys, os, shutil
sys.path.insert(0, '/tmp/piper/src/python')
os.environ['PYTHONPATH'] = '/tmp/piper/src/python'
from pathlib import Path

WORK = Path('/kaggle/working/iu-mienh')
OUTPUT = WORK / 'output'
PREPROC = WORK / 'preprocessed'

ckpts = sorted(OUTPUT.glob('**/*.ckpt'))
if not ckpts:
    print('No checkpoints found!')
else:
    ckpt = ckpts[-1]
    onnx_out = Path('/kaggle/working/iu-mienh.onnx')
    print(f'Exporting: {ckpt.name}')

    !PYTHONPATH=/tmp/piper/src/python python3 -m piper_train.export_onnx {ckpt} {onnx_out}

    # Copy config alongside model
    cfg_out = Path('/kaggle/working/iu-mienh.onnx.json')
    shutil.copy2(PREPROC / 'config.json', cfg_out)

    if onnx_out.exists():
        mb = onnx_out.stat().st_size / 1024 / 1024
        print(f'\n✅ Model exported: {mb:.1f} MB')
        print(f'Download from Output tab:')
        print(f'  iu-mienh.onnx')
        print(f'  iu-mienh.onnx.json')

In [ ]:
# Cell 7: Test inference
import sys, os
sys.path.insert(0, '/tmp/piper/src/python')
os.environ['PYTHONPATH'] = '/tmp/piper/src/python'
from pathlib import Path
import IPython.display as ipd

onnx = Path('/kaggle/working/iu-mienh.onnx')
cfg = Path('/kaggle/working/iu-mienh.onnx.json')

if onnx.exists():
    text = 'Tin-Hungh hnamv baamh mienh.'
    out = Path('/kaggle/working/test.wav')
    !echo "{text}" | PYTHONPATH=/tmp/piper/src/python python3 -m piper_train.infer_onnx --model {onnx} --config {cfg} --output-file {out}
    if out.exists():
        print(f'Generated: {text}')
        ipd.display(ipd.Audio(str(out)))
else:
    print('Export model first (Cell 6)')